## Imports

In [0]:
from pyspark.sql import SparkSession
import pandas as pd

run_date = dbutils.widgets.get("run_date")

In [0]:
##aws_utils
import boto3
from botocore.exceptions import NoCredentialsError

from pyspark.dbutils import DBUtils
from pyspark.sql import SparkSession
 
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix


In [0]:
env = get_env_prefix()

## Get the needed Perimeter and Date-Parameters

In [0]:
# Read the table into a Pandas DataFrame
Perimeter_path = f"`{env}-cdh`.reference.GeneralliPerimeter"

Perimeter = spark.read.table(Perimeter_path)
Perimeter.createOrReplaceTempView("Perimeter")

In [0]:
# The flow runs "today for yesterday's business date". When this EOM report is
# triggered by the orchestrator, yesterday's business date is, by construction, the
# last working day of its month. We therefore take PreviousBusinessDay of the run date
# (or today), which resolves to the correct month-end business date whether the
# triggering day falls in the next month or still inside the same month.
eomdate = spark.sql(f"""                    
                        select distinct  PreviousBusinessDay
                        from `{env}-cdh`.reference.ref_calendar
                        where CalendarDate = coalesce(try_cast({run_date!r} as date),cast(getdate() as date))
                    """).first()[0]

## Create Source Tables

In [0]:
DExposure = spark.sql(f"""                    
                        select distinct 
                                  AgreementKey
                                , ExposureKey
                                , coalesce(OnBalance, 0) OnBalance
                                , coalesce(OffBalance, 0) OffBalance
                        from `{env}-cdh`.derivative.DExposure
                        where BusinessDate = '{eomdate}'
                    """)

DExposure.createOrReplaceTempView("DExposure")

## Create and save the Final Table

In [0]:
Generalli_OUT = spark.sql(f"""                    
                            select distinct
                                  CAST(R.ReferenceDate AS DATE) ReferenceDate
                                , CAST(per.inAgreementNumber as STRING) AgreementNumber
                                , CAST(CONCAT(YEAR(ADD_MONTHS(LAST_DAY(r.referencedate), 1)), '-', MONTH(ADD_MONTHS(LAST_DAY(r.referencedate), 1)), '-', '3') AS DATE) FileProductionDate
                                , CAST(R.bal AS DECIMAL(10,2)) RemainingBalance
                                , CAST(NULL AS STRING) AccountStatus
                                , CAST(NULL AS STRING) DebtAccount
                                , CAST(NULL AS DATE) FirstInstallmentDate
                                , CAST(per.abCDI as STRING) CDI
                                , CAST(NULL AS STRING) Filler
                                , CAST(NULL AS DECIMAL) InstallementAmount
                                , CAST(NULL AS STRING) InsuranceNumber
                            from Perimeter per
                            inner join (
                                select p.id, p.assetkey, '{eomdate}' ReferenceDate, SUM(dex.OnBalance - dex.OffBalance) Bal
                                from Perimeter p
                                inner join DExposure dex
                                on P.AssetKey = dex.AgreementKey
                                group by all
                            ) R on per.id = R.id
                            order by CDI, AgreementNumber
                        """)

Generalli_OUT.createOrReplaceTempView("Generalli_OUT")

In [0]:
Output_path = f"`{env}-cdh-delivery`.it.Generalli_OUT"

# Save the DataFrame as a new table
Generalli_OUT.write.mode("overwrite").saveAsTable(Output_path)
